Montgomery County Maryland ABC Sales Data Dashboard

RUN FIRST CELL:
- Contains imports and setup for the entire notebook.
- Ensure proper connection to file by displaying head data.

In [101]:
import pandas as pd
import numpy as np
import requests
import PyPDF2
import io
import re
import sqlite3
import time
import datetime
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from difflib import SequenceMatcher
from openpyxl import Workbook

# Load CSV
df = pd.read_csv(r'C:\Users\LOW14\Downloads\Warehouse_and_Retail_Sales.csv')

More code to understand the data structure.

In [100]:
# Get a concise summary of the DataFrame
df.info()

# Get summary statistics for numerical columns
df.describe()

# List all column names
df.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 307645 entries, 0 to 307644
Data columns (total 9 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   YEAR              307645 non-null  int64  
 1   MONTH             307645 non-null  int64  
 2   SUPPLIER          307478 non-null  object 
 3   ITEM CODE         307645 non-null  object 
 4   ITEM DESCRIPTION  307645 non-null  object 
 5   ITEM TYPE         307644 non-null  object 
 6   RETAIL SALES      307642 non-null  float64
 7   RETAIL TRANSFERS  307645 non-null  float64
 8   WAREHOUSE SALES   307645 non-null  float64
dtypes: float64(3), int64(2), object(4)
memory usage: 21.1+ MB


Index(['YEAR', 'MONTH', 'SUPPLIER', 'ITEM CODE', 'ITEM DESCRIPTION',
       'ITEM TYPE', 'RETAIL SALES', 'RETAIL TRANSFERS', 'WAREHOUSE SALES'],
      dtype='object')

Clean and prepare data. First start with reviewing missing values.

In [3]:
df.isnull().sum()

YEAR                  0
MONTH                 0
SUPPLIER            167
ITEM CODE             0
ITEM DESCRIPTION      0
ITEM TYPE             1
RETAIL SALES          3
RETAIL TRANSFERS      0
WAREHOUSE SALES       0
dtype: int64

In [4]:
#review missing values from supplier column for further insite
missing_suppliers = df[df['SUPPLIER'].isnull()]
missing_suppliers.head()

,YEAR,MONTH,SUPPLIER,ITEM CODE,ITEM DESCRIPTION,ITEM TYPE,RETAIL SALES,RETAIL TRANSFERS,WAREHOUSE SALES
106,2020,1,NaN,107,JIGGER MEASURE SHOT GLASS,STR_SUPPLIES,14.69,18.0,0.0
188,2020,1,NaN,113,BARTENDERS BLACK BOOK,STR_SUPPLIES,0.40,0.0,0.0
231,2020,1,NaN,115,PLASTIC SHOT GLASS PACK,STR_SUPPLIES,5.71,6.0,0.0
252,2020,1,NaN,117,WHISKEY TASTING JOURNAL,STR_SUPPLIES,0.08,0.0,0.0
261,2020,1,NaN,118,PLASTIC WINE GLASS PACK,STR_SUPPLIES,7.40,10.0,0.0


In [5]:
missing_suppliers = df[df['ITEM TYPE'].isnull()]
missing_suppliers.head()

,YEAR,MONTH,SUPPLIER,ITEM CODE,ITEM DESCRIPTION,ITEM TYPE,RETAIL SALES,RETAIL TRANSFERS,WAREHOUSE SALES
96129,2017,10,REPUBLIC NATIONAL DISTRIBUTING CO,347939,FONTANAFREDDA BAROLO SILVER LABEL 750 ML,NaN,0.0,0.0,1.0


In [6]:
missing_suppliers = df[df['RETAIL SALES'].isnull()]
missing_suppliers.head()

,YEAR,MONTH,SUPPLIER,ITEM CODE,ITEM DESCRIPTION,ITEM TYPE,RETAIL SALES,RETAIL TRANSFERS,WAREHOUSE SALES
18390,2020,7,NaN,4,RMS ITEM,NON-ALCOHOL,NaN,0.0,0.0
299150,2020,9,NaN,3,COUPON,NON-ALCOHOL,NaN,0.0,0.0
300935,2020,9,NaN,4,RMS ITEM,NON-ALCOHOL,NaN,0.0,0.0


After reviewing the dataset, I plan to take the following steps to clean the data:

- Remove rows with missing values in the SUPPLIER and RETAIL SALES columns.
The missing SUPPLIER values appear mostly in non-alcoholic tertiary sales items (e.g., based on initial row inspection). Rows missing RETAIL SALES values appear to contain inconsequential or incomplete data.

- Address the single missing value in the ITEM TYPE column by manually assigning a value based on a review of the surrounding data.
Note: This decision is made for the purposes of portfolio demonstration. In a real-world setting, I would likely consult with my team or flag this item for further research. My experience in the wine industry suggests this may be a lower-volume item, which could justify additional scrutiny.

- Combine the YEAR and MONTH columns into a single DATE field to streamline time-based analysis.

Finally, I’m considering whether to continue in pandas or shift to a SQL-based workflow. A hybrid approach would better reflect real-world scenarios and demonstrate fluency in both environments. For this portfolio I will shift to a SQL-based workflow after data cleaning has been finished.

In [7]:
df_cleaned = df.dropna(subset=['SUPPLIER', 'RETAIL SALES']).copy() #drops any rows where supplier or retail sales are null values, and creates a copy to avoid future code warnings.

In [8]:
df_cleaned[df_cleaned['ITEM DESCRIPTION'].str.contains('FONTANAFREDDA BAROLO', case=False, na=False)]

,YEAR,MONTH,SUPPLIER,ITEM CODE,ITEM DESCRIPTION,ITEM TYPE,RETAIL SALES,RETAIL TRANSFERS,WAREHOUSE SALES
96129,2017,10,REPUBLIC NATIONAL DISTRIBUTING CO,347939,FONTANAFREDDA BAROLO SILVER LABEL 750 ML,NaN,0.0,0.0,1.0


In [9]:
df_cleaned.loc[96129, 'ITEM TYPE'] = 'WINE'

After reviewing the dataset, the item types listed in the file were very simple. This ITEM TYPE was converted to wine. Now that the data is cleaned, I will proceed to convert this into the SQL standard. 

In [10]:
# Load cleaned data into SQLite database
df_cleaned.to_sql('warehouse_sales', sqlite3.connect('warehouse_sales.db'), if_exists='replace', index=False)

307478

Create helper function for querying data.

In [11]:
def sql_query(query):
    """Helper function to run SQL queries and return pandas DataFrame"""
    conn = sqlite3.connect('warehouse_sales.db')
    result = pd.read_sql_query(query, conn)
    conn.close()
    return result

In [12]:
df_result = sql_query("""
SELECT *
FROM warehouse_sales 
LIMIT 10
""")
df_result

,YEAR,MONTH,SUPPLIER,ITEM CODE,ITEM DESCRIPTION,ITEM TYPE,RETAIL SALES,RETAIL TRANSFERS,WAREHOUSE SALES
0,2020,1,REPUBLIC NATIONAL DISTRIBUTING CO,100009,BOOTLEG RED - 750ML,WINE,0.00,0.0,2.0
1,2020,1,PWSWN INC,100024,MOMENT DE PLAISIR - 750ML,WINE,0.00,1.0,4.0
2,2020,1,RELIABLE CHURCHILL LLLP,1001,S SMITH ORGANIC PEAR CIDER - 18.7OZ,BEER,0.00,0.0,1.0
3,2020,1,LANTERNA DISTRIBUTORS INC,100145,SCHLINK HAUS KABINETT - 750ML,WINE,0.00,0.0,1.0
4,2020,1,DIONYSOS IMPORTS INC,100293,SANTORINI GAVALA WHITE - 750ML,WINE,0.82,0.0,0.0
5,2020,1,KYSELA PERE ET FILS LTD,100641,CORTENOVA VENETO P/GRIG - 750ML,WINE,2.76,0.0,6.0
6,2020,1,SANTA MARGHERITA USA INC,100749,SANTA MARGHERITA P/GRIG ALTO - 375ML,WINE,0.08,1.0,1.0
7,2020,1,BROWN-FORMAN BEVERAGES WORLDWIDE,1008,JACK DANIELS COUNTRY COCKTAIL SOUTHERN PEACH -...,BEER,0.00,0.0,2.0
8,2020,1,JIM BEAM BRANDS CO,10103,KNOB CREEK BOURBON 9YR - 100P - 375ML,LIQUOR,6.41,4.0,0.0
9,2020,1,INTERNATIONAL CELLARS LLC,101117,KSARA CAB - 750ML,WINE,0.33,1.0,2.0


After converting the file to SQL, I remembered to consolidate the date columns in the file to make future analysis easier. This additional cleaning step will be done at this point after SQL conversion. In a professional setting, I'd add this cleaning step earlier in the workflow with all the other cleaning steps. 

In [13]:
# Add the new DATE column to your existing df_cleaned
df_cleaned['DATE'] = pd.to_datetime(df_cleaned[['YEAR', 'MONTH']].assign(day=1))

In [14]:
#Verify new date column works
print("New DATE column sample:")
print(df_cleaned[['YEAR', 'MONTH', 'DATE']].head())
print(f"\nDATE column data type: {df_cleaned['DATE'].dtype}")

New DATE column sample:
   YEAR  MONTH       DATE
0  2020      1 2020-01-01
1  2020      1 2020-01-01
2  2020      1 2020-01-01
3  2020      1 2020-01-01
4  2020      1 2020-01-01

DATE column data type: datetime64[ns]


In [15]:
# Replace the database with the updated dataframe
df_cleaned.to_sql('warehouse_sales', sqlite3.connect('warehouse_sales.db'), if_exists='replace', index=False)

print("Database updated successfully!")

Database updated successfully!


In [16]:
# Test that the new DATE column works in SQL
df_test = sql_query("""
SELECT DATE, YEAR, MONTH, `RETAIL SALES`, `ITEM DESCRIPTION`
FROM warehouse_sales 
LIMIT 10
""")
df_test

,DATE,YEAR,MONTH,RETAIL SALES,ITEM DESCRIPTION
0,2020-01-01 00:00:00,2020,1,0.00,BOOTLEG RED - 750ML
1,2020-01-01 00:00:00,2020,1,0.00,MOMENT DE PLAISIR - 750ML
2,2020-01-01 00:00:00,2020,1,0.00,S SMITH ORGANIC PEAR CIDER - 18.7OZ
3,2020-01-01 00:00:00,2020,1,0.00,SCHLINK HAUS KABINETT - 750ML
4,2020-01-01 00:00:00,2020,1,0.82,SANTORINI GAVALA WHITE - 750ML
5,2020-01-01 00:00:00,2020,1,2.76,CORTENOVA VENETO P/GRIG - 750ML
6,2020-01-01 00:00:00,2020,1,0.08,SANTA MARGHERITA P/GRIG ALTO - 375ML
7,2020-01-01 00:00:00,2020,1,0.00,JACK DANIELS COUNTRY COCKTAIL SOUTHERN PEACH -...
8,2020-01-01 00:00:00,2020,1,6.41,KNOB CREEK BOURBON 9YR - 100P - 375ML
9,2020-01-01 00:00:00,2020,1,0.33,KSARA CAB - 750ML


In [17]:
winemagdf = pd.read_csv(r'C:\Users\LOW14\Downloads\winemag-data-130k-v2.csv.zip')

In [18]:
# Get a concise summary of the DataFrame
winemagdf.info()

# Get summary statistics for numerical columns
winemagdf.describe()

# List all column names
winemagdf.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129971 entries, 0 to 129970
Data columns (total 14 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Unnamed: 0             129971 non-null  int64  
 1   country                129908 non-null  object 
 2   description            129971 non-null  object 
 3   designation            92506 non-null   object 
 4   points                 129971 non-null  int64  
 5   price                  120975 non-null  float64
 6   province               129908 non-null  object 
 7   region_1               108724 non-null  object 
 8   region_2               50511 non-null   object 
 9   taster_name            103727 non-null  object 
 10  taster_twitter_handle  98758 non-null   object 
 11  title                  129971 non-null  object 
 12  variety                129970 non-null  object 
 13  winery                 129971 non-null  object 
dtypes: float64(1), int64(2), object(11)


Index(['Unnamed: 0', 'country', 'description', 'designation', 'points',
       'price', 'province', 'region_1', 'region_2', 'taster_name',
       'taster_twitter_handle', 'title', 'variety', 'winery'],
      dtype='object')

In [19]:
winemagdf.isnull().sum()

Unnamed: 0                   0
country                     63
description                  0
designation              37465
points                       0
price                     8996
province                    63
region_1                 21247
region_2                 79460
taster_name              26244
taster_twitter_handle    31213
title                        0
variety                      1
winery                       0
dtype: int64

In [20]:
missing_suppliers = winemagdf[winemagdf['region_1'].isnull()]
missing_suppliers.head()

,Unnamed: 0,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
1,1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
8,8,Germany,Savory dried thyme notes accent sunnier flavor...,Shine,87,12.0,Rheinhessen,NaN,NaN,Anna Lee C. Iijima,NaN,Heinz Eifel 2013 Shine Gewürztraminer (Rheinhe...,Gewürztraminer,Heinz Eifel
15,15,Germany,Zesty orange peels and apple notes abound in t...,Devon,87,24.0,Mosel,NaN,NaN,Anna Lee C. Iijima,NaN,Richard Böcking 2013 Devon Riesling (Mosel),Riesling,Richard Böcking
36,36,Chile,"White flower, lychee and apple aromas carry th...",Estate,86,15.0,Colchagua Valley,NaN,NaN,Michael Schachner,@wineschach,Estampa 2011 Estate Viognier-Chardonnay (Colch...,Viognier-Chardonnay,Estampa
44,44,Chile,A berry aroma comes with cola and herb notes. ...,NaN,86,9.0,Maule Valley,NaN,NaN,Michael Schachner,@wineschach,Sundance 2011 Merlot (Maule Valley),Merlot,Sundance


In [21]:
winemagdf_cleaned = winemagdf.dropna(subset=['country'])

In [22]:
winemagdf_cleaned.isnull().sum()

Unnamed: 0                   0
country                      0
description                  0
designation              37454
points                       0
price                     8992
province                     0
region_1                 21184
region_2                 79397
taster_name              26244
taster_twitter_handle    31213
title                        0
variety                      1
winery                       0
dtype: int64

In [23]:
# Testing sample search terms for new dataset
search_terms = ["SANTA MARGHERITA", "ROBERT MONDAVI", "CUPCAKE", "CAKEBREAD", "BUTTER"]

for term in search_terms:
    print(f"\n--- Searching for '{term}' ---")
    matches = winemagdf_cleaned[winemagdf_cleaned['title'].str.contains(term, case=False, na=False)]
    print(f"Found {len(matches)} matches")
    if len(matches) > 0:
        print("Sample matches:")
        for idx, row in matches.head(3).iterrows():
            print(f"  - {row['title']} | Country: {row['country']}")


--- Searching for 'SANTA MARGHERITA' ---
Found 10 matches
Sample matches:
  - Panizzi 2014 Vigna Santa Margherita  (Vernaccia di San Gimignano) | Country: Italy
  - Panizzi 2006 Vigna Santa Margherita  (Vernaccia di San Gimignano) | Country: Italy
  - Santa Margherita 2015 Extra Dry  (Valdobbiadene Prosecco Superiore) | Country: Italy

--- Searching for 'ROBERT MONDAVI' ---
Found 157 matches
Sample matches:
  - Robert Mondavi 2015 Fumé Blanc (Napa Valley) | Country: US
  - Robert Mondavi 2008 Cabernet Sauvignon (Napa Valley) | Country: US
  - Robert Mondavi 2011 Reserve Chardonnay (Carneros) | Country: US

--- Searching for 'CUPCAKE' ---
Found 31 matches
Sample matches:
  - Cupcake 2016 Rosé (California) | Country: US
  - Cupcake 2010 Cabernet Sauvignon (Central Coast) | Country: US
  - Cupcake 2012 Petite Sirah (Central Coast) | Country: US

--- Searching for 'CAKEBREAD' ---
Found 15 matches
Sample matches:
  - Cakebread 2014 Two Creeks Vineyards Pinot Noir (Anderson Valley) | Countr

After testing simple search results above, I will begin to create a search algorithm below. First up, testing a simple scoring function to differentiate between wines with similar names. 

In [102]:
def score_match(search_term, wine_title):
    """Score how well a search term matches a wine title."""
    
    # Convert to uppercase for case-insensitive comparison
    search_term = search_term.upper()
    wine_title = wine_title.upper()
    
    score = 0
    # Check if it's an exact producer match
    if wine_title.startswith(search_term):
        score += 100
    
    return score

def find_best_matches(search_term, winemagdf_cleaned, top_n=5):
    """
    Find the best matching wines for a search term and return them with scores
    """
    matches = []
    
    # First, find all wines that contain our search term
    potential_matches = winemagdf_cleaned[winemagdf_cleaned['title'].str.contains(search_term, case=False, na=False, regex=False)]
    
    # Now score each potential match
    for index, row in potential_matches.iterrows():
        score = score_match(search_term, row['title'])
        matches.append({
            'title': row['title'],
            'country': row['country'],
            'score': score
        })
    
    # Sort by score (highest first) and return top matches
    matches = sorted(matches, key=lambda x: x['score'], reverse=True)
    return matches[:top_n]

# Test the functions work together
print("Testing complete matching system:")
print("=" * 40)

test_terms = ["SANTA MARGHERITA", "ROBERT MONDAVI", "CUPCAKE"]
for term in test_terms:
    results = find_best_matches(term, winemagdf_cleaned, top_n=3)
    print(f"\nTop matches for {term}:")
    for match in results:
        print(f"  Score: {match['score']} | {match['title']} | Country: {match['country']}")

print("\n✅ Functions are working! Ready to proceed with country assignment.")

Testing complete matching system:

Top matches for SANTA MARGHERITA:
  Score: 100 | Santa Margherita 2015 Extra Dry  (Valdobbiadene Prosecco Superiore) | Country: Italy
  Score: 100 | Santa Margherita 2007 Pinot Grigio (Alto Adige) | Country: Italy
  Score: 100 | Santa Margherita 2002 Pinot Grigio (Valdadige) | Country: Italy

Top matches for ROBERT MONDAVI:
  Score: 100 | Robert Mondavi 2015 Fumé Blanc (Napa Valley) | Country: US
  Score: 100 | Robert Mondavi 2008 Cabernet Sauvignon (Napa Valley) | Country: US
  Score: 100 | Robert Mondavi 2011 Reserve Chardonnay (Carneros) | Country: US

Top matches for CUPCAKE:
  Score: 100 | Cupcake 2016 Rosé (California) | Country: US
  Score: 100 | Cupcake 2010 Cabernet Sauvignon (Central Coast) | Country: US
  Score: 100 | Cupcake 2012 Petite Sirah (Central Coast) | Country: US

✅ Functions are working! Ready to proceed with country assignment.


In [25]:
def extract_producer_candidates(wine_description):
    """Extract potential producer names from ABC wine descriptions"""
    if pd.isna(wine_description):
        return []
        
    # Remove common terms that aren't producer names
    terms_to_remove = ['750ML', '375ML', '1.5L', 'RED', 'WHITE', 'WINE', 'P/GRIG', 'CAB', 'SAUV', 'CHARD']
    
    # Clean the description
    cleaned = str(wine_description).upper()
    for term in terms_to_remove:
        cleaned = cleaned.replace(term, '').replace('-', ' ').strip()
    
    words = [word.strip() for word in cleaned.split() if len(word.strip()) > 1]
    
    candidates = []
    # Try different combinations
    if len(words) >= 1:
        candidates.append(words[0])
    if len(words) >= 2:
        candidates.append(f"{words[0]} {words[1]}")
    if len(words) >= 3:
        candidates.append(f"{words[0]} {words[1]} {words[2]}")
    
    return candidates

def find_country_for_wine(wine_description, winemagdf_cleaned):
    """Find the country for a wine from MOCO data using Wine Magazine database"""
    # Get producer candidates
    candidates = extract_producer_candidates(wine_description)
    
    best_result = None
    highest_confidence = 0
    best_matched_row = None
    
    for candidate in candidates:
        # Find wines by this producer in the database
        matches = find_best_matches(candidate, winemagdf_cleaned, top_n=10)
        
        if matches and len(matches) > 0:
            # Get the most common country for wines with good scores
            good_matches = [m for m in matches if m['score'] >= 50]
            
            if good_matches:
                # Get the best match (highest score)
                best_match = max(good_matches, key=lambda x: x['score'])
                
                countries = [m['country'] for m in good_matches]
                # Find most common country
                country_counts = {}
                for country in countries:
                    country_counts[country] = country_counts.get(country, 0) + 1
                
                most_common_country = max(country_counts, key=country_counts.get)
                confidence = len(good_matches)
                
                if confidence > highest_confidence:
                    # Find the actual row in the dataframe for the best match
                    matched_row_data = None
                    if 'index' in best_match:  # If your matches include the dataframe index
                        matched_row_data = winemagdf_cleaned.loc[best_match['index']].to_dict()
                    elif 'row' in best_match:  # If your matches include the actual row data
                        matched_row_data = best_match['row']
                    
                    best_result = {
                        'country': most_common_country,
                        'producer_found': candidate,
                        'confidence': confidence,
                        'total_matches': len(matches),
                        'matched_row': matched_row_data  # This is what was missing!
                    }
                    highest_confidence = confidence
                    best_matched_row = matched_row_data
    
    return best_result
# Test with a few wines from your actual dataset first
print("Testing with actual MOCO wines:")
print("=" * 50)

# Filter for just wine items
wine_items = df_cleaned[df_cleaned['ITEM TYPE'] == 'WINE'].copy()
print(f"Found {len(wine_items)} wine items in your dataset")

# Test with first 5 wines
test_wines = wine_items['ITEM DESCRIPTION'].head(5).tolist()

for wine in test_wines:
    print(f"\nWine: {wine}")
    result = find_country_for_wine(wine, winemagdf_cleaned)
    if result:
        print(f"  Country: {result['country']}")
        print(f"  Producer: {result['producer_found']}")
        print(f"  Confidence: {result['confidence']} matching wines")
    else:
        print("  No match found")

print("\n" + "=" * 50)
print("If results look good, run the full processing with:")
print("df_with_countries = add_countries_to_moco_data(df_cleaned, winemagdf_cleaned)")

def add_countries_to_moco_data(df_cleaned, winemagdf_cleaned):
    """Add country information to MOCO dataset"""
    print(f"Processing {len(df_cleaned)} items...")
    
    # Create copy and initialize new columns
    df_with_countries = df_cleaned.copy()
    df_with_countries['COUNTRY'] = 'Unknown'
    df_with_countries['PRODUCER_FOUND'] = ''
    df_with_countries['MATCH_CONFIDENCE'] = 0
    
    # Only process wine items
    wine_mask = df_with_countries['ITEM TYPE'] == 'WINE'
    wine_items = df_with_countries[wine_mask]
    
    print(f"Found {len(wine_items)} wine items to process...")
    
    for idx, row in wine_items.iterrows():
        wine_description = row['ITEM DESCRIPTION']
        
        # Find country
        result = find_country_for_wine(wine_description, winemagdf_cleaned)
        
        if result:
            df_with_countries.at[idx, 'COUNTRY'] = result['country']
            df_with_countries.at[idx, 'PRODUCER_FOUND'] = result['producer_found']
            df_with_countries.at[idx, 'MATCH_CONFIDENCE'] = result['confidence']
        
        # Progress update every 100 items
        if (idx - wine_items.index[0]) % 100 == 0:
            processed = idx - wine_items.index[0] + 1
            print(f"Processed {processed}/{len(wine_items)} wines...")
    
    return df_with_countries

print("\nReady to process your MOCO dataset!")

Testing with actual MOCO wines:
Found 187641 wine items in your dataset

Wine: BOOTLEG RED - 750ML
  Country: US
  Producer: BOOTLEG
  Confidence: 2 matching wines

Wine: MOMENT DE PLAISIR - 750ML
  No match found

Wine: SCHLINK HAUS KABINETT - 750ML
  Country: Germany
  Producer: SCHLINK
  Confidence: 2 matching wines

Wine: SANTORINI GAVALA WHITE - 750ML
  No match found

Wine: CORTENOVA VENETO P/GRIG - 750ML
  Country: Italy
  Producer: CORTENOVA
  Confidence: 3 matching wines

If results look good, run the full processing with:
df_with_countries = add_countries_to_moco_data(df_cleaned, winemagdf_cleaned)

Ready to process your MOCO dataset!


Now that the matching processes has been tested, The below script will match through the entire document. I'm moving this cell to markdown after complete to save resources in the future. 

In [26]:
def add_all_match_columns(df_cleaned, winemagdf_cleaned):
    """
    Fast version using producer-candidate matching and caching.
    Includes multiple columns from the match result.
    """
    print("Starting country/enrichment matching with caching...")

    df_with_matches = df_cleaned.copy()
    df_with_matches['COUNTRY'] = 'Unknown'
    df_with_matches['PRODUCER_FOUND'] = ''
    df_with_matches['MATCH_CONFIDENCE'] = 0
    df_with_matches['TOTAL_MATCHES'] = 0

    description_cache = {}
    matched_count = 0
    cache_hits = 0

    wine_mask = df_with_matches['ITEM TYPE'] == 'WINE'
    wine_items = df_with_matches[wine_mask]

    print(f"Found {len(wine_items)} wine items to process...")

    for idx, row in wine_items.iterrows():
        desc = row['ITEM DESCRIPTION']

        # Check cache first
        if desc in description_cache:
            result = description_cache[desc]
            cache_hits += 1
        else:
            result = find_country_for_wine(desc, winemagdf_cleaned)
            description_cache[desc] = result

        if result:
            df_with_matches.at[idx, 'COUNTRY'] = result.get('country', 'Unknown')
            df_with_matches.at[idx, 'PRODUCER_FOUND'] = result.get('producer_found', '')
            df_with_matches.at[idx, 'MATCH_CONFIDENCE'] = result.get('confidence', 0)
            df_with_matches.at[idx, 'TOTAL_MATCHES'] = result.get('total_matches', 0)
            matched_count += 1

        # Progress every 100
        if (idx - wine_items.index[0]) % 100 == 0:
            processed = idx - wine_items.index[0] + 1
            cache_eff = (cache_hits / processed) * 100 if processed else 0
            print(f"Processed {processed:,} wines")
            print(f"  Matches: {matched_count:,} ({matched_count/processed*100:.1f}%)")
            print(f"  Cache hits: {cache_hits:,} ({cache_eff:.1f}% efficiency)")
            print(f"  Unique descriptions processed: {len(description_cache):,}")

    print("\n" + "="*60)
    print("FINAL RESULTS:")
    print(f"Total wines processed: {len(wine_items):,}")
    print(f"Matches found: {matched_count:,}")
    print(f"Cache efficiency: {cache_hits/len(wine_items)*100:.1f}%")
    print("Done.")

    return df_with_matches

#df_with_matches = add_all_match_columns(df_cleaned, winemagdf_cleaned)

# Or save as pickle for faster loading (preserves data types)
df_with_matches.to_pickle('wine_dataset_with_matches_COMPLETE.pkl')

In [27]:
# Load your saved dataset
print("Loading your enhanced wine dataset...")
df_with_matches = pd.read_pickle('wine_dataset_with_countries_COMPLETE.pkl')

print("✅ Dataset loaded successfully!")
print(f"📊 Total rows: {len(df_with_matches):,}")
print(f"📋 Total columns: {len(df_with_matches.columns)}")

# Check the structure 
print(f"\n📁 Column names:")
for i, col in enumerate(df_with_matches.columns):
    print(f"  {i+1}. {col}")

# Verify your country data is intact
print(f"\n🌍 Country column analysis:")
print(f"  Total wines with countries: {df_with_matches['country'].notna().sum():,}")
print(f"  Total wines without countries: {df_with_matches['country'].isna().sum():,}")
print(f"  Match rate: {(df_with_matches['country'].notna().sum() / len(df_with_matches) * 100):.1f}%")

# Show the country distribution
print(f"\n🏆 Top 10 countries by wine count:")
country_counts = df_with_matches['country'].value_counts().head(10)
for country, count in country_counts.items():
    print(f"  {country}: {count:,} wines")

# Quick data sample
print(f"\n👀 Sample of your data:")
print(df_with_matches[['ITEM DESCRIPTION', 'country']].head())

# Verify data types
print(f"\n🔧 Data types:")
print(df_with_matches.dtypes)

Loading your enhanced wine dataset...
✅ Dataset loaded successfully!
📊 Total rows: 307,478
📋 Total columns: 11

📁 Column names:
  1. YEAR
  2. MONTH
  3. SUPPLIER
  4. ITEM CODE
  5. ITEM DESCRIPTION
  6. ITEM TYPE
  7. RETAIL SALES
  8. RETAIL TRANSFERS
  9. WAREHOUSE SALES
  10. DATE
  11. country

🌍 Country column analysis:
  Total wines with countries: 181,074
  Total wines without countries: 126,404
  Match rate: 58.9%

🏆 Top 10 countries by wine count:
  US: 97,630 wines
  Italy: 21,982 wines
  France: 20,281 wines
  Spain: 8,536 wines
  Australia: 6,587 wines
  Chile: 4,633 wines
  Argentina: 3,917 wines
  Portugal: 3,914 wines
  New Zealand: 3,438 wines
  South Africa: 2,903 wines

👀 Sample of your data:
                      ITEM DESCRIPTION  country
0                  BOOTLEG RED - 750ML       US
1            MOMENT DE PLAISIR - 750ML      NaN
2  S SMITH ORGANIC PEAR CIDER - 18.7OZ       US
3        SCHLINK HAUS KABINETT - 750ML  Germany
4       SANTORINI GAVALA WHITE - 750ML

After running the above dataset, I caught an error. The original "find_country_for_wine" function was not written to transpose over all columns data. I will begin a revised version of this funtion below. 

In [28]:
def add_all_match_columns(df_cleaned, winemagdf_cleaned):
    """
    Fast version using producer-candidate matching and caching.
    Includes ALL columns from the winemag dataset for matching rows.
    """
    print("Starting country/enrichment matching with caching...")
    df_with_matches = df_cleaned.copy()
    
    # Get all column names from winemag dataset
    winemag_columns = winemagdf_cleaned.columns.tolist()
    print(f"Will add {len(winemag_columns)} columns from winemag dataset: {winemag_columns}")
    
    # Initialize all winemag columns in the main dataframe
    for col in winemag_columns:
        # Avoid overwriting existing columns - add prefix if needed
        if col in df_with_matches.columns:
            new_col_name = f"winemag_{col}"
            print(f"Column '{col}' already exists, using '{new_col_name}'")
        else:
            new_col_name = col
        
        # Set appropriate default values based on likely column types
        if col.lower() in ['country']:
            df_with_matches[new_col_name] = 'Unknown'
        elif col.lower() in ['points', 'price', 'score', 'rating']:
            df_with_matches[new_col_name] = 0
        else:
            df_with_matches[new_col_name] = ''  # Default for text columns
    
    # Keep your existing tracking columns
    df_with_matches['PRODUCER_FOUND'] = ''
    df_with_matches['MATCH_CONFIDENCE'] = 0
    df_with_matches['TOTAL_MATCHES'] = 0
    
    description_cache = {}
    matched_count = 0
    cache_hits = 0
    
    wine_mask = df_with_matches['ITEM TYPE'] == 'WINE'
    wine_items = df_with_matches[wine_mask]
    print(f"Found {len(wine_items)} wine items to process...")
    
    # Add progress tracking for long-running process
    total_wines = len(wine_items)
    start_time = time.time()
    
    for i, (idx, row) in enumerate(wine_items.iterrows()):
        desc = row['ITEM DESCRIPTION']
        
        # Check cache first
        if desc in description_cache:
            result = description_cache[desc]
            cache_hits += 1
        else:
            result = find_country_for_wine(desc, winemagdf_cleaned)
            description_cache[desc] = result
        
        if result:
            # Add all columns from the matching winemag row
            if 'matched_row' in result and result['matched_row'] is not None:
                matched_row = result['matched_row']
                for col in winemag_columns:
                    # Use same naming logic as initialization
                    if col in df_cleaned.columns:
                        new_col_name = f"winemag_{col}"
                    else:
                        new_col_name = col
                    
                    # Get value from matched row, with fallback
                    value = matched_row.get(col, '')
                    df_with_matches.at[idx, new_col_name] = value
            
            # Keep your existing metadata columns
            df_with_matches.at[idx, 'PRODUCER_FOUND'] = result.get('producer_found', '')
            df_with_matches.at[idx, 'MATCH_CONFIDENCE'] = result.get('confidence', 0)
            df_with_matches.at[idx, 'TOTAL_MATCHES'] = result.get('total_matches', 0)
            matched_count += 1
        
        # Add checkpointing every 1000 items
        if (i + 1) % 1000 == 0:
            checkpoint_filename = f'checkpoint_{i+1}.pkl'
            df_with_matches.to_pickle(checkpoint_filename)
            print(f"*** CHECKPOINT SAVED: {checkpoint_filename} ***")
        
        # Enhanced progress reporting for long-running process
        if (i + 1) % 100 == 0 or (i + 1) == total_wines:
            processed = i + 1
            elapsed_time = time.time() - start_time
            cache_eff = (cache_hits / processed) * 100 if processed else 0
            
            # Calculate ETA
            if processed > 0:
                avg_time_per_item = elapsed_time / processed
                remaining_items = total_wines - processed
                eta_seconds = remaining_items * avg_time_per_item
                eta = str(datetime.timedelta(seconds=int(eta_seconds)))
            else:
                eta = "Unknown"
            
            print(f"Processed {processed:,}/{total_wines:,} wines ({processed/total_wines*100:.1f}%)")
            print(f"  Matches: {matched_count:,} ({matched_count/processed*100:.1f}%)")
            print(f"  Cache hits: {cache_hits:,} ({cache_eff:.1f}% efficiency)")
            print(f"  Unique descriptions: {len(description_cache):,}")
            print(f"  Elapsed: {str(datetime.timedelta(seconds=int(elapsed_time)))}")
            print(f"  ETA: {eta}")
            print("-" * 40)
     
    total_time = time.time() - start_time
    print("\n" + "="*60)
    print("FINAL RESULTS:")
    print(f"Total wines processed: {total_wines:,}")
    print(f"Matches found: {matched_count:,} ({matched_count/total_wines*100:.1f}%)")
    print(f"Cache efficiency: {cache_hits/total_wines*100:.1f}%")
    print(f"Total runtime: {str(datetime.timedelta(seconds=int(total_time)))}")
    print(f"Average time per wine: {total_time/total_wines:.3f} seconds")
    print(f"Columns added from winemag: {len(winemag_columns)}")
    print("Done.")
    
    return df_with_matches

In [29]:
# Run the function
df_with_matches = add_all_match_columns(df_cleaned, winemagdf_cleaned)

Starting country/enrichment matching with caching...
Will add 14 columns from winemag dataset: ['Unnamed: 0', 'country', 'description', 'designation', 'points', 'price', 'province', 'region_1', 'region_2', 'taster_name', 'taster_twitter_handle', 'title', 'variety', 'winery']
Found 187641 wine items to process...
Processed 100/187,641 wines (0.1%)
  Matches: 73 (73.0%)
  Cache hits: 0 (0.0% efficiency)
  Unique descriptions: 100
  Elapsed: 0:00:39
  ETA: 20:35:36
----------------------------------------
Processed 200/187,641 wines (0.1%)
  Matches: 147 (73.5%)
  Cache hits: 0 (0.0% efficiency)
  Unique descriptions: 200
  Elapsed: 0:01:24
  ETA: 21:55:22
----------------------------------------
Processed 300/187,641 wines (0.2%)
  Matches: 218 (72.7%)
  Cache hits: 0 (0.0% efficiency)
  Unique descriptions: 300
  Elapsed: 0:01:56
  ETA: 20:12:55
----------------------------------------
Processed 400/187,641 wines (0.2%)
  Matches: 294 (73.5%)
  Cache hits: 0 (0.0% efficiency)
  Unique d

In [57]:
# Check the shape of your DataFrames
print("Original df shape:", df_cleaned.shape)
print("Original winemagdf shape:", winemagdf_cleaned.shape)
print("Result df shape:", df_with_matches.shape)

# Check if there are any new columns
original_columns = set(df_cleaned.columns)
new_columns = set(df_with_matches.columns)
added_columns = new_columns - original_columns
print("Added columns:", list(added_columns))

# Look at a few sample rows
print(df_with_matches.head())

Original df shape: (307478, 10)
Original winemagdf shape: (129908, 14)
Result df shape: (307478, 27)
Added columns: ['title', 'designation', 'description', 'Unnamed: 0', 'PRODUCER_FOUND', 'region_1', 'region_2', 'TOTAL_MATCHES', 'taster_twitter_handle', 'winery', 'MATCH_CONFIDENCE', 'price', 'variety', 'points', 'province', 'country', 'taster_name']
   YEAR  MONTH                           SUPPLIER ITEM CODE  \
0  2020      1  REPUBLIC NATIONAL DISTRIBUTING CO    100009   
1  2020      1                          PWSWN INC    100024   
2  2020      1            RELIABLE CHURCHILL LLLP      1001   
3  2020      1          LANTERNA DISTRIBUTORS INC    100145   
4  2020      1               DIONYSOS IMPORTS INC    100293   

                      ITEM DESCRIPTION ITEM TYPE  RETAIL SALES  \
0                  BOOTLEG RED - 750ML      WINE          0.00   
1            MOMENT DE PLAISIR - 750ML      WINE          0.00   
2  S SMITH ORGANIC PEAR CIDER - 18.7OZ      BEER          0.00   
3    

In [58]:
dbm = pd.read_csv(r'C:\Users\LOW14\Downloads\Distributors_Virginia_Three_Main.csv')

In [59]:
# Get a concise summary of the DataFrame
dbm.info()

# Get summary statistics for numerical columns
dbm.describe()

# List all column names
dbm.columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14279 entries, 0 to 14278
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   License_ID    14279 non-null  int64 
 1   Brand_Name    14279 non-null  object
 2   Product_Type  14279 non-null  object
 3   Distributor   14279 non-null  object
dtypes: int64(1), object(3)
memory usage: 446.3+ KB


Index(['License_ID', 'Brand_Name', 'Product_Type', 'Distributor'], dtype='object')

In [60]:
dbm.isnull().sum()

License_ID      0
Brand_Name      0
Product_Type    0
Distributor     0
dtype: int64

In [61]:
dbm_cleaned = dbm.copy()

In [62]:
# Load cleaned data into SQLite database
dbm_cleaned.to_sql('Distributor_List', sqlite3.connect('distributor_list.db'), if_exists='replace', index=False)

14279

In [63]:
from my_data_tools.validator import validate_dataframe, quick_validate

# Test that it worked
print("✅ Data validation tools loaded successfully!")

✅ Data validation tools loaded successfully!


In [64]:
# Your existing code
dbm_cleaned = dbm.copy()

# Run the validation
validate_dataframe(dbm_cleaned)

# Or for a quick check
quick_validate(dbm_cleaned)

UNIVERSAL DATA VALIDATION REPORT

BASIC OVERVIEW
--------------------
Shape: 14,279 rows × 4 columns
Memory usage: 0.4 MB
Column names: ['License_ID', 'Brand_Name', 'Product_Type', 'Distributor']

DATA TYPES
--------------------
object    3
int64     1
Name: count, dtype: int64

MISSING VALUES
--------------------
No missing values found!

DUPLICATE ANALYSIS
--------------------
Complete duplicate rows: 5,132
Sample duplicates:
   License_ID             Brand_Name Product_Type Distributor
0       85629              #NICEWINE         Wine        RNDC
1       85629                 10SPAN         Wine        RNDC
2       85629         12 GENERATIONS         Wine        RNDC
3       85629             12 KNIGHTS         Wine        RNDC
4       85629  12 WINES OF CHRISTMAS         Wine        RNDC
5       85629             13 CELSIUS         Wine        RNDC
6       85629               14 HANDS         Wine        RNDC
7       85629                   1749         Wine        RNDC
8       85

In [65]:
def clean_text_for_matching(text):
    """
    Clean text for better matching by removing common business suffixes,
    standardizing spacing, and converting to lowercase
    """
    if pd.isna(text):
        return ""
    
    # Convert to string and lowercase
    text = str(text).lower().strip()
    
    # Remove common business suffixes/prefixes
    business_terms = [
        r'\b(inc|llc|ltd|corp|corporation|company|co|wines|winery|vineyards|estate|cellars|bros|brothers)\b\.?',
        r'\b(the|le|la|di|del|della|von|van)\b',  # Articles and prepositions
        r'[&\+]',  # Ampersands and plus signs
        r'[\.\,\-\']',  # Punctuation
    ]
    
    for pattern in business_terms:
        text = re.sub(pattern, ' ', text)
    
    # Clean up extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

def calculate_similarity(text1, text2):
    """
    Calculate similarity between two text strings using multiple methods
    Returns a score between 0 and 1
    """
    if not text1 or not text2:
        return 0
    
    # Method 1: SequenceMatcher (overall similarity)
    seq_similarity = SequenceMatcher(None, text1, text2).ratio()
    
    # Method 2: Word overlap (how many words are shared)
    words1 = set(text1.split())
    words2 = set(text2.split())
    
    if not words1 or not words2:
        word_overlap = 0
    else:
        word_overlap = len(words1.intersection(words2)) / max(len(words1), len(words2))
    
    # Method 3: Substring matching (is one contained in the other?)
    substring_match = 0
    if text1 in text2 or text2 in text1:
        substring_match = 0.3  # Bonus for substring matches
    
    # Combined score (weighted average)
    combined_score = (seq_similarity * 0.5) + (word_overlap * 0.4) + substring_match
    
    return min(combined_score, 1.0)  # Cap at 1.0

def match_suppliers_to_distributors(sales_df, distributor_df, 
                                  supplier_col='SUPPLIER', 
                                  brand_col='Brand_Name', 
                                  distributor_col='Distributor',
                                  similarity_threshold=0.4,
                                  max_matches_per_supplier=5):
    """
    Match suppliers from sales data to distributors based on brand names
    
    Parameters:
    - sales_df: DataFrame with supplier information
    - distributor_df: DataFrame with brand names and distributors
    - supplier_col: Column name for suppliers in sales_df
    - brand_col: Column name for brand names in distributor_df
    - distributor_col: Column name for distributors in distributor_df
    - similarity_threshold: Minimum similarity score for a match (0-1)
    - max_matches_per_supplier: Maximum number of distributor matches per supplier
    
    Returns:
    - DataFrame with matching results and statistics
    """
    
    print(f"Starting supplier-distributor matching...")
    print(f"Sales data: {len(sales_df):,} rows with {sales_df[supplier_col].nunique():,} unique suppliers")
    print(f"Distributor data: {len(distributor_df):,} rows with {distributor_df[brand_col].nunique():,} unique brands")
    print(f"Similarity threshold: {similarity_threshold}")
    print("-" * 50)
    
    # Get unique suppliers and brands
    unique_suppliers = sales_df[supplier_col].dropna().unique()
    unique_brands = distributor_df[brand_col].dropna().unique()
    
    # Clean text for matching
    print("Cleaning text for matching...")
    supplier_cleaned = {supplier: clean_text_for_matching(supplier) for supplier in unique_suppliers}
    brand_cleaned = {brand: clean_text_for_matching(brand) for brand in unique_brands}
    
    # Find matches
    print(f"Finding matches for {len(unique_suppliers):,} suppliers...")
    matches = []
    
    for i, supplier in enumerate(unique_suppliers):
        if i % 100 == 0:  # Progress indicator
            print(f"  Processed {i:,}/{len(unique_suppliers):,} suppliers...")
        
        supplier_clean = supplier_cleaned[supplier]
        supplier_matches = []
        
        # Compare with all brands
        for brand in unique_brands:
            brand_clean = brand_cleaned[brand]
            similarity = calculate_similarity(supplier_clean, brand_clean)
            
            if similarity >= similarity_threshold:
                # Get all distributors for this brand
                brand_distributors = distributor_df[distributor_df[brand_col] == brand][distributor_col].unique()
                
                for distributor in brand_distributors:
                    supplier_matches.append({
                        'supplier_original': supplier,
                        'supplier_cleaned': supplier_clean,
                        'brand_original': brand,
                        'brand_cleaned': brand_clean,
                        'distributor': distributor,
                        'similarity_score': similarity
                    })
        
        # Sort matches by similarity and keep top matches
        supplier_matches.sort(key=lambda x: x['similarity_score'], reverse=True)
        matches.extend(supplier_matches[:max_matches_per_supplier])
    
    # Convert to DataFrame
    matches_df = pd.DataFrame(matches)
    
    if len(matches_df) == 0:
        print("No matches found! Try lowering the similarity threshold.")
        return pd.DataFrame()
    
    # Print summary statistics
    print(f"\nMATCHING RESULTS:")
    print(f"Total matches found: {len(matches_df):,}")
    print(f"Suppliers with matches: {matches_df['supplier_original'].nunique():,}")
    print(f"Suppliers without matches: {len(unique_suppliers) - matches_df['supplier_original'].nunique():,}")
    print(f"Average similarity score: {matches_df['similarity_score'].mean():.3f}")
    
    # Show suppliers with multiple distributor matches
    multiple_matches = matches_df.groupby('supplier_original').size()
    suppliers_with_multiple = multiple_matches[multiple_matches > 1]
    
    if len(suppliers_with_multiple) > 0:
        print(f"\nSuppliers with multiple distributor matches: {len(suppliers_with_multiple):,}")
        print("Top 5 suppliers with most matches:")
        print(suppliers_with_multiple.sort_values(ascending=False).head().to_string())
    
    return matches_df

def add_distributor_to_sales(sales_df, matches_df, 
                           supplier_col='SUPPLIER',
                           how='best_match'):
    """
    Add distributor information to the sales DataFrame
    
    Parameters:
    - sales_df: Original sales DataFrame
    - matches_df: Results from match_suppliers_to_distributors()
    - supplier_col: Column name for suppliers
    - how: 'best_match' (highest similarity), 'all_matches' (comma-separated), or 'first_match'
    
    Returns:
    - Enhanced sales DataFrame with distributor information
    """
    
    sales_enhanced = sales_df.copy()
    
    if how == 'best_match':
        # Keep only the best match per supplier
        best_matches = matches_df.groupby('supplier_original').apply(
            lambda x: x.loc[x['similarity_score'].idxmax()]
        ).reset_index(drop=True)
        
        # Create mapping dictionary
        distributor_map = dict(zip(best_matches['supplier_original'], best_matches['distributor']))
        similarity_map = dict(zip(best_matches['supplier_original'], best_matches['similarity_score']))
        
        # Add to sales data
        sales_enhanced['Distributor'] = sales_enhanced[supplier_col].map(distributor_map)
        sales_enhanced['Match_Confidence'] = sales_enhanced[supplier_col].map(similarity_map)
        
    elif how == 'all_matches':
        # Combine all matches per supplier (comma-separated)
        all_matches = matches_df.groupby('supplier_original').agg({
            'distributor': lambda x: ', '.join(sorted(set(x))),
            'similarity_score': 'max'
        }).reset_index()
        
        # Create mapping dictionary
        distributor_map = dict(zip(all_matches['supplier_original'], all_matches['distributor']))
        similarity_map = dict(zip(all_matches['supplier_original'], all_matches['similarity_score']))
        
        # Add to sales data
        sales_enhanced['Distributors_All'] = sales_enhanced[supplier_col].map(distributor_map)
        sales_enhanced['Match_Confidence'] = sales_enhanced[supplier_col].map(similarity_map)
        
    elif how == 'first_match':
        # Keep first match per supplier
        first_matches = matches_df.groupby('supplier_original').first().reset_index()
        
        # Create mapping dictionary
        distributor_map = dict(zip(first_matches['supplier_original'], first_matches['distributor']))
        similarity_map = dict(zip(first_matches['supplier_original'], first_matches['similarity_score']))
        
        # Add to sales data
        sales_enhanced['Distributor'] = sales_enhanced[supplier_col].map(distributor_map)
        sales_enhanced['Match_Confidence'] = sales_enhanced[supplier_col].map(similarity_map)
    
    # Summary
    matched_rows = sales_enhanced['Distributor' if 'Distributor' in sales_enhanced.columns else 'Distributors_All'].notna().sum()
    print(f"\nENHANCED SALES DATA:")
    print(f"Total rows: {len(sales_enhanced):,}")
    print(f"Rows with distributor match: {matched_rows:,} ({matched_rows/len(sales_enhanced)*100:.1f}%)")
    
    return sales_enhanced

# USAGE EXAMPLE:
# Step 1: Your data is already loaded as:
# df_with_matches = DataFrame with 'SUPPLIER' column
# dbm_cleaned = DataFrame with 'Brand_Name' and 'Distributor' columns

# Step 2: Find matches
# matches = match_suppliers_to_distributors(df_with_matches, dbm_cleaned)

# Step 3: Review matches before adding to sales data
# print(matches.head(10))

# Step 4: Add distributor info to sales data
# sales_with_distributors = add_distributor_to_sales(df_with_matches, matches, how='best_match')

# QUICK VALIDATION FUNCTION
def review_matches(matches_df, n_samples=10):
    """
    Quick function to review the quality of matches
    """
    print("SAMPLE MATCHES FOR REVIEW:")
    print("-" * 50)
    
    sample = matches_df.sample(min(n_samples, len(matches_df)))
    
    for _, row in sample.iterrows():
        print(f"Supplier: '{row['supplier_original']}'")
        print(f"  -> Brand: '{row['brand_original']}'")
        print(f"  -> Distributor: '{row['distributor']}'")
        print(f"  -> Similarity: {row['similarity_score']:.3f}")
        print()

# Usage: review_matches(matches)

In [66]:
# Step 1: Find matches between your DataFrames
matches = match_suppliers_to_distributors(
    df_with_matches,    # Has the 'SUPPLIER' column
    dbm_cleaned,        # Has 'Brand_Name' and 'Distributor' columns
    supplier_col='SUPPLIER',
    brand_col='Brand_Name', 
    distributor_col='Distributor',
    similarity_threshold=0.4
)

# Step 2: Review some matches to check quality
review_matches(matches, n_samples=5)

# Step 3: Add distributors to your sales data
df_enhanced = add_distributor_to_sales(
    df_with_matches, 
    matches, 
    how='best_match'  # or 'all_matches' for multiple distributors
)

Starting supplier-distributor matching...
Sales data: 307,478 rows with 396 unique suppliers
Distributor data: 14,279 rows with 8,095 unique brands
Similarity threshold: 0.4
--------------------------------------------------
Cleaning text for matching...
Finding matches for 396 suppliers...
  Processed 0/396 suppliers...
  Processed 100/396 suppliers...
  Processed 200/396 suppliers...
  Processed 300/396 suppliers...

MATCHING RESULTS:
Total matches found: 1,112
Suppliers with matches: 323
Suppliers without matches: 73
Average similarity score: 0.523

Suppliers with multiple distributor matches: 254
Top 5 suppliers with most matches:
supplier_original
LCF WINE COMPANY LLC         5
PHILLIPS FARMS LLC           5
O'NEILL BEVERAGES CO LLC     5
OCEAN CITY BREWING CO LLC    5
ONE TRUE VINE                5
SAMPLE MATCHES FOR REVIEW:
--------------------------------------------------
Supplier: 'NOBLE VINTNERS INC'
  -> Brand: 'LE NOBLE'
  -> Distributor: 'RNDC'
  -> Similarity: 0.763

Sup

In [67]:
# Get a concise summary of the DataFrame
df_enhanced.info()

# Get summary statistics for numerical columns
df_enhanced.describe()

# List all column names
df_enhanced.columns

<class 'pandas.core.frame.DataFrame'>
Index: 307478 entries, 0 to 307644
Data columns (total 29 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   YEAR                   307478 non-null  int64         
 1   MONTH                  307478 non-null  int64         
 2   SUPPLIER               307478 non-null  object        
 3   ITEM CODE              307478 non-null  object        
 4   ITEM DESCRIPTION       307478 non-null  object        
 5   ITEM TYPE              307478 non-null  object        
 6   RETAIL SALES           307478 non-null  float64       
 7   RETAIL TRANSFERS       307478 non-null  float64       
 8   WAREHOUSE SALES        307478 non-null  float64       
 9   DATE                   307478 non-null  datetime64[ns]
 10  Unnamed: 0             307478 non-null  object        
 11  country                307478 non-null  object        
 12  description            307478 non-null  object   

Index(['YEAR', 'MONTH', 'SUPPLIER', 'ITEM CODE', 'ITEM DESCRIPTION',
       'ITEM TYPE', 'RETAIL SALES', 'RETAIL TRANSFERS', 'WAREHOUSE SALES',
       'DATE', 'Unnamed: 0', 'country', 'description', 'designation', 'points',
       'price', 'province', 'region_1', 'region_2', 'taster_name',
       'taster_twitter_handle', 'title', 'variety', 'winery', 'PRODUCER_FOUND',
       'MATCH_CONFIDENCE', 'TOTAL_MATCHES', 'Distributor', 'Match_Confidence'],
      dtype='object')

In [68]:
df_enhanced = df_enhanced.copy()

In [82]:
import sqlite3

conn = sqlite3.connect('df_enhanced.db')
cursor = conn.cursor()
cursor.execute("DROP TABLE IF EXISTS Df_Enhanced;")
conn.commit()

df_enhanced.to_sql('Df_Enhanced', conn, if_exists='replace', index=False)

conn.commit()
conn.close()


In [81]:
df_enhanced = df_enhanced.drop(columns=['MATCH_CONFIDENCE'])

In [88]:
def sql_query(query):
    """Helper function to run SQL queries and return pandas DataFrame"""
    conn = sqlite3.connect('df_enhanced.db')
    result = pd.read_sql_query(query, conn)
    conn.close()
    return result

In [94]:

df_result = sql_query("""
SELECT *
FROM Df_Enhanced
WHERE [variety] LIKE '%White Blend%'
LIMIT 50
""")
df_result

,YEAR,MONTH,SUPPLIER,ITEM CODE,ITEM DESCRIPTION,ITEM TYPE,RETAIL SALES,RETAIL TRANSFERS,WAREHOUSE SALES,DATE,...,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery,PRODUCER_FOUND,TOTAL_MATCHES,Distributor


In [96]:
empty_count = (df_enhanced['variety'].str.strip() == '').sum()
print(f"Number of empty 'variety' entries: {empty_count}")

Number of empty 'variety' entries: 307478
